In [1]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../data/raw/cicids_ddos.csv")

df.columns = df.columns.str.strip()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

# Remove identifiers
columns_to_drop = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp"
]

df = df.drop(
    columns=[col for col in columns_to_drop if col in df.columns],
    errors="ignore"
)

# Encode labels
df["Label"] = df["Label"].map({
    "BENIGN": 0,
    "DDoS": 1
})

df = df.dropna(subset=["Label"])

X = df.drop("Label", axis=1)
y = df["Label"].astype(int)

print("Features:", X.shape)
print("Labels:", y.shape)

Features: (225711, 78)
Labels: (225711,)


In [3]:
constant_columns = X.columns[X.nunique() <= 1]

print("Constant features:", len(constant_columns))
print(list(constant_columns))

X = X.drop(columns=constant_columns)

Constant features: 10
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (180568, 68)
Testing: (45143, 68)


In [5]:
selector = SelectKBest(
    score_func=f_classif,
    k=min(30, X_train.shape[1])
)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Selected features:")
for feature in selected_features:
    print(feature)

Selected features:
Destination Port
Total Length of Fwd Packets
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Packets/s
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Fwd PSH Flags
Min Packet Length
Max Packet Length
Packet Length Mean
Packet Length Std
Packet Length Variance
SYN Flag Count
PSH Flag Count
URG Flag Count
Down/Up Ratio
Average Packet Size
Avg Fwd Segment Size
Avg Bwd Segment Size
Subflow Fwd Bytes
min_seg_size_forward


In [6]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

xgb_model.fit(
    X_train_selected,
    y_train
)

training_time = time.time() - start_time

print("Training time:", training_time, "seconds")

Training time: 0.8842978477478027 seconds


In [7]:
y_pred = xgb_model.predict(X_test_selected)
y_prob = xgb_model.predict_proba(X_test_selected)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

Accuracy: 0.9998
Precision: 0.9999
Recall: 0.9998
F1: 0.9998
ROC-AUC: 1.0000


In [8]:
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

In [9]:
base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="recall",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

random_search.fit(
    X_train_selected,
    y_train
)

tuning_time = time.time() - start_time

print("Tuning time:", tuning_time)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Tuning time: 35.79891490936279


In [10]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV Recall:")
print(random_search.best_score_)

Best parameters:
{'subsample': 1.0, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Best CV Recall:
0.9996875610232375


In [11]:
best_xgb = random_search.best_estimator_

y_pred = best_xgb.predict(X_test_selected)
y_prob = best_xgb.predict_proba(X_test_selected)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.9998227853709324
Precision: 0.999882826231301
Recall   : 0.9998047256395235
F1 Score : 0.9998437744102484
ROC-AUC  : 0.9999991034849967


In [12]:
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(best_xgb, "../models/best_ids_model.pkl")
joblib.dump(selector, "../models/feature_selector.pkl")

print("Model saved successfully.")

Model saved successfully.
